In [ ]:
import os
import sys
import glob
import numpy as np
from tqdm import tqdm
import torch
import torch.multiprocessing
torch.multiprocessing.set_sharing_strategy('file_system')
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

from neural_methods.model.PhysMamba import PhysMamba
from neural_methods.loss.PhysNetNegPearsonLoss import Neg_Pearson

In [ ]:
# ----- paths -----
PREPROCESSED_PATH = os.path.join(REPO_ROOT, "preprocessed_data/Headmotion/groupE")
SAVE_MODEL_DIR = os.path.join(REPO_ROOT, "final_model_release")
SAVE_MODEL_PATH = os.path.join(SAVE_MODEL_DIR, "GroupE_PhysMamba.pth")

# ----- params -----
CHUNK_LENGTH = 128     # frames per clip
BATCH_SIZE = 4
EPOCHS = 10
LR = 1e-4

# ----- device -----
DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

os.makedirs(SAVE_MODEL_DIR, exist_ok=True)

In [ ]:
# Dataset and DataLoader
all_input_files = glob.glob(os.path.join(PREPROCESSED_PATH, "*", "*_input*.npy"))
print(f"Found {len(all_input_files)} preprocessed clips.")

class PhysMambaDataset(Dataset):
    def __init__(self, input_files):
        self.inputs = sorted(input_files)
        self.labels = [f.replace("_input", "_label") for f in self.inputs]

    def __len__(self):
        return len(self.inputs)

    def __getitem__(self, index):
        data  = np.float32(np.load(self.inputs[index]))   # (D, H, W, 3)
        label = np.float32(np.load(self.labels[index]))   # (D,)

        # NDHWC -> NCDHW: transpose to (3, D, H, W)
        data = np.transpose(data, (3, 0, 1, 2))
        return data, label

dataset = PhysMambaDataset(all_input_files)
train_loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=4, drop_last=True)
print(f"DataLoader ready: {len(train_loader)} batches")

In [ ]:
# Model, Loss, Optimizer
model = PhysMamba(frames=CHUNK_LENGTH).to(DEVICE)
criterion = Neg_Pearson()
optimizer = optim.Adam(model.parameters(), lr=LR, weight_decay=0.0005)
scheduler = torch.optim.lr_scheduler.OneCycleLR(optimizer, max_lr=LR, epochs=EPOCHS, steps_per_epoch=max(1, len(train_loader)))


In [ ]:
# Training loop
for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0
    tbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}")
    
    for idx, (data, labels) in enumerate(tbar):
        data = data.to(DEVICE)
        labels = labels.to(DEVICE)
        
        optimizer.zero_grad()
        
        # Forward pass
        pred_ppg, _, _, _ = model(data)
        
        # Normalize predictions per clip
        pred_ppg = (pred_ppg - torch.mean(pred_ppg, axis=-1).view(-1, 1)) / (torch.std(pred_ppg, axis=-1).view(-1, 1) + 1e-7)
        
        # Normalize labels
        labels = (labels - torch.mean(labels)) / (torch.std(labels) + 1e-7)
        
        loss = criterion(pred_ppg, labels)
        loss.backward()
        
        optimizer.step()
        scheduler.step()
        
        running_loss += loss.item()
        tbar.set_postfix({'loss': loss.item()})
        
    epoch_loss = running_loss / max(1, len(train_loader))
    print(f"Epoch {epoch+1} completed. Average Loss: {epoch_loss:.4f}")

print("Training complete.")

In [ ]:
# Save model weights
torch.save(model.state_dict(), SAVE_MODEL_PATH)
print(f"Saved model weights to {SAVE_MODEL_PATH}")